# MTUS1 tertile analysis across additional tumour cohorts

## Overview

This notebook applies the *MTUS1*-low versus *MTUS1*-high tertile comparison to additional TCGA tumour cohorts and pooled datasets. These analyses are exploratory and are used to evaluate whether the TNBC-derived pathway pattern is observed beyond breast cancer.


In [ ]:
import scanpy as sc
import decoupler as dc
from anndata import AnnData
import random

# Import DESeq2
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.preprocessing import deseq2_norm,deseq2_norm_fit,deseq2_norm_transform

# Only needed for processing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans

from sklearn.metrics import silhouette_score

from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind

# batch correction
from inmoose.pycombat import pycombat_seq

from conorm import tmm,mrn,getmm

#from train import *
import sys
sys.path.append("..")  # Adds the parent directory to the system path
from src import utils
from src.utils import RNAseq_DataSet,pool_dataset

%load_ext autoreload
%autoreload 2


In [ ]:
# Analysis settings
name_gene = "MTUS1"
condition_a_tester = ["condition", "MTUS1-", "MTUS1+"]
condition_cancer = "all_cancer"
name_TCGA = "TCGA-BRCA"


# 1. Load Data

In [ ]:
def load_bc_all():

    VUMC = RNAseq_DataSet(['VUMC'])
    # TNBC only
    VUMC.load_data()
    VUMC.group = VUMC.metadata

    SRP157974 = RNAseq_DataSet(['SRP157974'])
    # TNBC only
    SRP157974.load_data()
    SRP157974.group = SRP157974.metadata

    GSE181466 = RNAseq_DataSet(['GSE181466'])
    # TNBC only
    GSE181466.load_data()
    GSE181466.group = GSE181466.metadata

    SRP042620 = RNAseq_DataSet(['SRP042620'])
    SRP042620.load_data()
    SRP042620 = SRP042620.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE','ER_ENRICHED']}})
    SRP042620.group = SRP042620.metadata

    GSE192341 = RNAseq_DataSet(['GSE192341'])
    GSE192341.load_data()
    #GSE192341 = GSE192341.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'PCR': ['PCR', 'noPCR']}})
    GSE192341.group = GSE192341.metadata

    #GSE123845 = RNAseq_DataSet(['GSE123845'])
    #GSE123845.load_data()
    #GSE123845 = GSE123845.get_smaller_subset({'genes': None, 'patients': {'TIME': ['DIAGNOSIS']}})
    #GSE123845.group = GSE123845.metadata

    GSE202203 = RNAseq_DataSet(['GSE202203'])
    GSE202203.load_data()
    GSE202203.features = GSE202203.features.loc[GSE202203.metadata.index]

    TCGA = RNAseq_DataSet([name_TCGA])
    TCGA.load_data()
    TCGA = TCGA.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})
    TCGA.group = TCGA.metadata

    #VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA = pool_dataset({ 'VUMC':VUMC,'SRP042620':SRP042620,'SRP157974': SRP157974, 'GSE181466':GSE181466, 'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA')
    #VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA.group = VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA.metadata
    datasets = {'SRP042620':SRP042620,'GSE192341':GSE192341, 'TCGA': TCGA, 'GSE202203': GSE202203}#, 'VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA': VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA}
    datasets = {'SRP042620': SRP042620,  'VUMC':VUMC, 'GSE192341':GSE192341, 'GSE181466':GSE181466,'TCGA': TCGA, 'GSE202203': GSE202203, 'SRP157974':SRP157974}
    #VUMC_GSE192341_GSE123845_GSE202203_TCGA = pool_dataset({ 'VUMC':VUMC, 'GSE123845': GSE123845, 'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_GSE192341_GSE123845_GSE202203_TCGA')
    #VUMC_GSE192341_GSE123845_GSE202203_TCGA.group = VUMC_GSE192341_GSE123845_GSE202203_TCGA.metadata
    #datasets = {'VUMC':VUMC, 'GSE192341':GSE192341,'GSE123845': GSE123845,  'GSE202203': GSE202203, 'TCGA': TCGA,'VUMC_GSE192341_GSE123845_GSE202203_TCGA': VUMC_GSE192341_GSE123845_GSE202203_TCGA}
    return datasets

def load_tnbc_only():
        VUMC = RNAseq_DataSet(['VUMC'])
        # TNBC only
        VUMC.load_data()
        VUMC.group = VUMC.metadata

        SRP157974 = RNAseq_DataSet(['SRP157974'])
        # TNBC only
        SRP157974.load_data()
        SRP157974.group = SRP157974.metadata

        GSE181466 = RNAseq_DataSet(['GSE181466'])
        # TNBC only
        GSE181466.load_data()
        GSE181466.group = GSE181466.metadata

        # NO because MTUS1 is low for all samples in this dataset
        #GSE163882 = RNAseq_DataSet(['GSE163882'])
        #GSE163882.load_data()
        #GSE163882 = GSE163882.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'HER2_FROM_IHC': ['HER2_LOW'], 'PR_FROM_IHC': ['PR_LOW']}})
        #GSE163882.group = GSE163882.metadata

        # No beccause PDX models only and no patient data
        #GSE235167 = RNAseq_DataSet(['GSE235167'])
        #GSE235167.load_data()
        #GSE235167 = GSE235167.get_smaller_subset({'genes': None, 'patients': SUBTYPE': ['TRIPLE_NEGATIVE']}})
        #GSE235167.group = GSE235167.metadata

        SRP042620 = RNAseq_DataSet(['SRP042620'])
        SRP042620.load_data()
        SRP042620 = SRP042620.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE',]}})
        SRP042620.group = SRP042620.metadata

        # seulement 6 samples TNBC
        #SRP032789 = RNAseq_DataSet(['SRP032789'])
        #SRP032789.load_data()
        #SRP032789 = SRP032789.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE']}})
        #SRP032789.group = SRP032789.metadata

        # Do not keep RSEM values.
        #GSE123845 = RNAseq_DataSet(['GSE123845'])
        #GSE123845.load_data()
        #GSE123845 = GSE123845.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE'],'TIME': ['DIAGNOSIS']}})
        #GSE123845.group = GSE123845.metadata

        GSE192341 = RNAseq_DataSet(['GSE192341'])
        GSE192341.load_data()
        GSE192341 = GSE192341.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW']}})
        GSE192341.group = GSE192341.metadata
        
    
        GSE202203 = RNAseq_DataSet(['GSE202203'])
        GSE202203.load_data()
        GSE202203.features = GSE202203.features.loc[GSE202203.metadata.index]
        GSE202203 = GSE202203.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE']}})
        GSE202203.group = GSE202203.metadata

        TCGA = RNAseq_DataSet(['TCGA-BRCA'])
        TCGA.load_data()
        TCGA = TCGA.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'HER2_FROM_IHC': ['HER2_LOW'], 'PR_FROM_IHC': ['PR_LOW'], 'sample_type': ['Primary Tumor']}})
        TCGA.group = TCGA.metadata

        # without GSE123845
        #VUMC_GSE192341_GSE202203_TCGA = pool_dataset({ 'VUMC':VUMC,  'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_GSE192341_GSE123845_GSE202203_TCGA')
        #VUMC_GSE192341_GSE202203_TCGA.group = VUMC_GSE192341_GSE202203_TCGA.metadata
        #datasets = {'VUMC':VUMC, 'GSE192341':GSE192341,  'GSE202203': GSE202203, 'TCGA': TCGA,'VUMC_GSE192341_GSE202203_TCGA': VUMC_GSE192341_GSE202203_TCGA}
        
        # without GSE123845 and with SRP157974
        #VUMC_SRP157974_GSE192341_GSE202203_TCGA = pool_dataset({ 'VUMC':VUMC, 'SRP157974':SRP157974,  'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_SRP157974_GSE192341_GSE202203_TCGA')
        #VUMC_SRP157974_GSE192341_GSE202203_TCGA.group = VUMC_SRP157974_GSE192341_GSE202203_TCGA.metadata
        #datasets = {'VUMC':VUMC, 'SRP157974':SRP157974, 'GSE192341':GSE192341,  'GSE202203': GSE202203, 'TCGA': TCGA,'VUMC_SRP157974_GSE192341_GSE202203_TCGA': VUMC_SRP157974_GSE192341_GSE202203_TCGA }


        #VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA = pool_dataset({'GSE181466':GSE181466, 'SRP157974':SRP157974, 'VUMC':VUMC, 'SRP042620': SRP042620, 'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA')
        #VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA.group = VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA.metadata
        datasets = {'SRP042620': SRP042620,  'VUMC':VUMC, 'GSE192341':GSE192341, 'GSE181466':GSE181466,'TCGA': TCGA, 'GSE202203': GSE202203, 'SRP157974':SRP157974}#, 'VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA': VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA}
        return datasets

def load_all_cancer():
    
    TCGA_LIHC = RNAseq_DataSet(['TCGA-LIHC']) #?
    TCGA_LIHC.load_data()
    TCGA_LIHC = TCGA_LIHC.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_LUAD = RNAseq_DataSet(['TCGA-LUAD']) #?
    TCGA_LUAD.load_data()
    TCGA_LUAD = TCGA_LUAD.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_LUSC = RNAseq_DataSet(['TCGA-LUSC']) #oui
    TCGA_LUSC.load_data()
    TCGA_LUSC = TCGA_LUSC.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_LGG = RNAseq_DataSet(['TCGA-LGG']) #?
    TCGA_LGG.load_data()
    TCGA_LGG = TCGA_LGG.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_COAD = RNAseq_DataSet(['TCGA-COAD']) #ok
    TCGA_COAD.load_data()
    TCGA_COAD = TCGA_COAD.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_PAAD = RNAseq_DataSet(['TCGA-PAAD']) #?
    TCGA_PAAD.load_data()
    TCGA_PAAD = TCGA_PAAD.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_OV = RNAseq_DataSet(['TCGA-OV']) #?
    TCGA_OV.load_data()
    TCGA_OV = TCGA_OV.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    #TCGA_CHOL = RNAseq_DataSet(['TCGA-CHOL']) #NON
    #TCGA_CHOL.load_data()
    #TCGA_CHOL = TCGA_CHOL.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    #####
    #TCGA_KICH = RNAseq_DataSet(['TCGA-KICH'])#NON
    #TCGA_KICH.load_data()
    #TCGA_KICH = TCGA_KICH.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_KIRP = RNAseq_DataSet(['TCGA-KIRP'])#OK
    TCGA_KIRP.load_data()
    TCGA_KIRP = TCGA_KIRP.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_KIRC = RNAseq_DataSet(['TCGA-KIRC'])#ok
    TCGA_KIRC.load_data()
    TCGA_KIRC = TCGA_KIRC.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_BLCA = RNAseq_DataSet(['TCGA-BLCA']) #ok
    TCGA_BLCA.load_data()
    TCGA_BLCA = TCGA_BLCA.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_HNSC = RNAseq_DataSet(['TCGA-HNSC'])#?
    TCGA_HNSC.load_data()
    TCGA_HNSC = TCGA_HNSC.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    #TCGA_THCA = RNAseq_DataSet(['TCGA-THCA']) #NON
    #TCGA_THCA.load_data()
    #TCGA_THCA = TCGA_THCA.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_PRAD = RNAseq_DataSet(['TCGA-PRAD']) #?
    TCGA_PRAD.load_data()
    TCGA_PRAD = TCGA_PRAD.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_READ = RNAseq_DataSet(['TCGA-READ']) #OK
    TCGA_READ.load_data()
    TCGA_READ = TCGA_READ.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_UCEC = RNAseq_DataSet(['TCGA-UCEC']) #OK
    TCGA_UCEC.load_data()
    TCGA_UCEC = TCGA_UCEC.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    #TCGA_STAD = RNAseq_DataSet(['TCGA-STAD']) #NON
    #TCGA_STAD.load_data()
    #TCGA_STAD = TCGA_STAD.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    # BC
    VUMCa = RNAseq_DataSet(['VUMC'])
    # TNBC only
    VUMCa.load_data()
    VUMCa.group = VUMCa.metadata

    SRP157974a = RNAseq_DataSet(['SRP157974'])
    # TNBC only
    SRP157974a.load_data()
    SRP157974a.group = SRP157974a.metadata

    GSE181466a = RNAseq_DataSet(['GSE181466'])
    # TNBC only
    GSE181466a.load_data()
    GSE181466a.group = GSE181466a.metadata

    SRP042620a = RNAseq_DataSet(['SRP042620'])
    SRP042620a.load_data()
    SRP042620a = SRP042620a.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE','ER_ENRICHED']}})
    SRP042620a.group = SRP042620a.metadata

    GSE192341a = RNAseq_DataSet(['GSE192341'])
    GSE192341a.load_data()
    GSE192341a.group = GSE192341a.metadata

    #GSE123845 = RNAseq_DataSet(['GSE123845'])
    #GSE123845.load_data()
    #GSE123845 = GSE123845.get_smaller_subset({'genes': None, 'patients': {'TIME': ['DIAGNOSIS']}})
    #GSE123845.group = GSE123845.metadata

    GSE202203a = RNAseq_DataSet(['GSE202203'])
    GSE202203a.load_data()
    GSE202203a.features = GSE202203a.features.loc[GSE202203a.metadata.index]

    TCGAa = RNAseq_DataSet([name_TCGA])
    TCGAa.load_data()
    TCGAa = TCGAa.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})
    TCGAa.group = TCGAa.metadata

    # TN
    VUMC = RNAseq_DataSet(['VUMC'])
    # TNBC only
    VUMC.load_data()
    VUMC.group = VUMC.metadata

    SRP157974 = RNAseq_DataSet(['SRP157974'])
    # TNBC only
    SRP157974.load_data()
    SRP157974.group = SRP157974.metadata

    GSE181466 = RNAseq_DataSet(['GSE181466'])
    # TNBC only
    GSE181466.load_data()
    GSE181466.group = GSE181466.metadata

    SRP042620 = RNAseq_DataSet(['SRP042620'])
    SRP042620.load_data()
    SRP042620 = SRP042620.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE',]}})
    SRP042620.group = SRP042620.metadata

    GSE192341 = RNAseq_DataSet(['GSE192341'])
    GSE192341.load_data()
    GSE192341 = GSE192341.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW']}})
    GSE192341.group = GSE192341.metadata
    

    GSE202203 = RNAseq_DataSet(['GSE202203'])
    GSE202203.load_data()
    GSE202203.features = GSE202203.features.loc[GSE202203.metadata.index]
    GSE202203 = GSE202203.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE']}})
    GSE202203.group = GSE202203.metadata

    TCGA = RNAseq_DataSet(['TCGA-BRCA'])
    TCGA.load_data()
    TCGA = TCGA.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'HER2_FROM_IHC': ['HER2_LOW'], 'PR_FROM_IHC': ['PR_LOW'], 'sample_type': ['Primary Tumor']}})
    TCGA.group = TCGA.metadata

    #datasets = {'TCGA-LUAD': TCGA_LUAD, 'TCGA-LUSC': TCGA_LUSC,'TCGA-KIRC': TCGA_KIRC,'TCGA-COAD': TCGA_COAD, 'TCGA-PAAD': TCGA_PAAD, 'TCGA-OV': TCGA_OV,  'GSE123845_GSE202203_TCGA': BC_all_pooled, 'VUMC_GSE192341_GSE123845_GSE202203_TCGA': TNBC_all_pooled }
    datasets = {'TCGA-KIRC': TCGA_KIRC,'TCGA-KIRP':TCGA_KIRP,'TCGA-BLCA':TCGA_BLCA,'TCGA-COAD': TCGA_COAD,'TCGA-READ':TCGA_READ, 'TCGA-PAAD': TCGA_PAAD, 'TCGA-LIHC':TCGA_LIHC,'TCGA-LGG':TCGA_LGG ,'TCGA-HNSC':TCGA_HNSC,'TCGA-LUAD': TCGA_LUAD, 'TCGA-LUSC': TCGA_LUSC,'TCGA-PRAD':TCGA_PRAD,  'TCGA-OV': TCGA_OV,'TCGA-UCEC': TCGA_UCEC,
                'VUMCa':VUMCa, 'SRP157974a':SRP157974a, 'GSE181466a':GSE181466a, 'SRP042620a': SRP042620a, 'GSE192341a':GSE192341a, 'GSE202203a': GSE202203a, 'TCGAa': TCGAa,    
                'VUMC':VUMC, 'SRP157974':SRP157974, 'GSE181466':GSE181466, 'SRP042620': SRP042620, 'GSE192341':GSE192341,  'GSE202203': GSE202203, 'TCGA': TCGA}#, 'VUMCa_GSE192341a_GSE123845a_GSE202203a_TCGAa': BC_all_pooled, 'VUMC_GSE192341_GSE123845_GSE202203_TCGA': TNBC_all_pooled } #'TCGA-CHOL':TCGA_CHOL,'TCGA-STAD':TCGA_STAD,'TCGA-THCA':TCGA_THCA,'TCGA-KICH':TCGA_KICH,
    return datasets



def load_bc_and_tn_separately():
    # BC
    
    VUMCa = RNAseq_DataSet(['VUMC'])
    # TNBC only
    VUMCa.load_data()
    VUMCa.group = VUMCa.metadata

    SRP157974a = RNAseq_DataSet(['SRP157974'])
    # TNBC only
    SRP157974a.load_data()
    SRP157974a.group = SRP157974a.metadata

    GSE181466a = RNAseq_DataSet(['GSE181466'])
    # TNBC only
    GSE181466a.load_data()
    GSE181466a.group = GSE181466a.metadata

    SRP042620a = RNAseq_DataSet(['SRP042620'])
    SRP042620a.load_data()
    SRP042620a = SRP042620a.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE','ER_ENRICHED']}})
    SRP042620a.group = SRP042620a.metadata

    GSE192341a = RNAseq_DataSet(['GSE192341'])
    GSE192341a.load_data()
    #GSE192341 = GSE192341.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'PCR': ['PCR', 'noPCR']}})
    GSE192341a.group = GSE192341a.metadata

    #GSE123845 = RNAseq_DataSet(['GSE123845'])
    #GSE123845.load_data()
    #GSE123845 = GSE123845.get_smaller_subset({'genes': None, 'patients': {'TIME': ['DIAGNOSIS']}})
    #GSE123845.group = GSE123845.metadata

    GSE202203a = RNAseq_DataSet(['GSE202203'])
    GSE202203a.load_data()
    GSE202203a.features = GSE202203a.features.loc[GSE202203a.metadata.index]

    TCGAa = RNAseq_DataSet([name_TCGA])
    TCGAa.load_data()
    TCGAa = TCGAa.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})
    TCGAa.group = TCGAa.metadata

    #BC_all_pooled = pool_dataset({ 'SRP042620a': SRP042620a,  'VUMCa':VUMCa, 'GSE192341a':GSE192341a, 'GSE181466a':GSE181466a,'TCGAa': TCGAa, 'GSE202203a': GSE202203a, 'SRP157974a':SRP157974a},'VUMCa_GSE192341a_GSE181466a_SRP042620a_GSE202203a_TCGAa_SRP157974a')

    # TN
    VUMC = RNAseq_DataSet(['VUMC'])
    # TNBC only
    VUMC.load_data()
    VUMC.group = VUMC.metadata

    SRP157974 = RNAseq_DataSet(['SRP157974'])
    # TNBC only
    SRP157974.load_data()
    SRP157974.group = SRP157974.metadata

    GSE181466 = RNAseq_DataSet(['GSE181466'])
    # TNBC only
    GSE181466.load_data()
    GSE181466.group = GSE181466.metadata

    SRP042620 = RNAseq_DataSet(['SRP042620'])
    SRP042620.load_data()
    SRP042620 = SRP042620.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE',]}})
    SRP042620.group = SRP042620.metadata

    GSE192341 = RNAseq_DataSet(['GSE192341'])
    GSE192341.load_data()
    GSE192341 = GSE192341.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW']}})
    GSE192341.group = GSE192341.metadata
    

    GSE202203 = RNAseq_DataSet(['GSE202203'])
    GSE202203.load_data()
    GSE202203.features = GSE202203.features.loc[GSE202203.metadata.index]
    GSE202203 = GSE202203.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE']}})
    GSE202203.group = GSE202203.metadata

    TCGA = RNAseq_DataSet(['TCGA-BRCA'])
    TCGA.load_data()
    TCGA = TCGA.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'HER2_FROM_IHC': ['HER2_LOW'], 'PR_FROM_IHC': ['PR_LOW'], 'sample_type': ['Primary Tumor']}})
    TCGA.group = TCGA.metadata

    #TNBC_all_pooled = pool_dataset({ 'VUMC':VUMC, 'GSE123845': GSE123845, 'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_GSE192341_GSE123845_GSE202203_TCGA')
    #TNBC_all_pooled.group = TNBC_all_pooled.metadata

    datasets = {'SRP042620a': SRP042620a,  'VUMCa':VUMCa, 'GSE192341a':GSE192341a, 'GSE181466a':GSE181466a,'TCGAa': TCGAa, 'GSE202203a': GSE202203a, 'SRP157974a':SRP157974a,'SRP042620': SRP042620,  'VUMC':VUMC, 'GSE192341':GSE192341, 'GSE181466':GSE181466,'TCGA': TCGA, 'GSE202203': GSE202203, 'SRP157974':SRP157974}#, 'BC_all_pooled': BC_all_pooled,'TNBC_all_pooled': TNBC_all_pooled}
    return datasets


if condition_cancer == "allBC":
    datasets = load_bc_all()
elif condition_cancer == "TN":
    datasets = load_tnbc_only()
elif condition_cancer == "all_cancer":
    datasets = load_all_cancer()
elif condition_cancer == "allBC+TN":
    datasets = load_bc_and_tn_separately()
else:
    raise ValueError(f"Unknown condition_cancer value: {condition_cancer}")

In [ ]:
for name,dst in datasets.items():
    print(name, ", number of samples: ", (dst.features.shape[0]))

# 2. Make 2 groups , differential analysis (Gene, GSEA, TF)

## A. Tertile-based comparison groups

In [ ]:
for i, (name_dataset,dataset) in enumerate(datasets.items()):
    
    #try : 
        print(name_dataset)
        if 1:
            
            Raw_counts = dataset.features.round().astype(int)
            # Normalization
            normalize_counts_df = deseq2_norm(dataset.features)[0]
            vst_df = np.log2(normalize_counts_df + 1)

            t1 = vst_df[name_gene].quantile(1/3)
            t2 = vst_df[name_gene].quantile(2/3)
            dataset.metadata = dataset.metadata.copy()
            dataset.metadata['condition_transfer'] = [condition_a_tester[2] if x > t2 else (condition_a_tester[1] if x<t1 else 'unknown') for x in vst_df[name_gene]]

            # Create anndata object
            dataset.anndata = AnnData(X=dataset.features.astype(int), obs=dataset.metadata)
            dataset.anndata.obs['condition'] = dataset.metadata['condition_transfer']
            adata = dataset.anndata

            # QUALITY CONTROL Obtain genes that pass the thresholds
            genes = dc.filter_by_expr(adata, group='condition', min_count=1, min_total_count=10, large_n=1, min_prop=1)
            # Filter by these genes
            adata = adata[:, genes].copy()
            dataset.anndata = adata
            print("*"*20)


In [ ]:
def pool_dataset_2(dict_dataset, pool_dataset_name):
    """
    Pool several datasets while retaining all metadata fields.
    - Concatenate count matrices by rows.
    - Concatenate metadata tables while automatically aligning columns.
    - Add a 'dataset' column to record the source dataset for each sample.
    """

    dst = RNAseq_DataSet([pool_dataset_name]) 
    Raw_counts = None
    metadata_ = pd.DataFrame()

    for dataset_name, dataset in dict_dataset.items():

        # --- 1) Counts ---
        counts = dataset.features.copy()
        counts = counts.round().astype(int)

        # Keep only shared genes.
        if Raw_counts is None:
            Raw_counts = counts
        else:
            common_genes = Raw_counts.columns.intersection(counts.columns)
            Raw_counts = Raw_counts[common_genes]
            counts = counts[common_genes]
            Raw_counts = pd.concat([Raw_counts, counts], axis=0)

        # --- 2) Metadata ---
        meta = dataset.metadata.copy()

        # Add source dataset information.
        meta["dataset"] = dataset_name

        # Concatenate while automatically aligning columns.
        metadata_ = pd.concat([metadata_, meta], axis=0, sort=False)

    # finalisation
    dst.features = Raw_counts
    dst.metadata = metadata_

    dst.common_predictors = list(dst.features.columns)
    dst.dataset_type = "RNA-seq:counts"

    # Create anndata object
    dst.metadata = dst.metadata.loc[dst.features.index].copy()
    dst.anndata = AnnData(X=dst.features.astype(int), obs=dst.metadata)
    dst.anndata.obs["condition_transfer"] = dst.anndata.obs["condition_transfer"].astype(str)
    dst.anndata.obs["condition"] = dst.anndata.obs["condition_transfer"]
    # QUALITY CONTROL Obtain genes that pass the thresholds
    genes = dc.filter_by_expr(dst.anndata, group='condition', min_count=1, min_total_count=10, large_n=1, min_prop=1)
    # Filter by these genes
    dst.anndata = dst.anndata[:, genes].copy()

    return dst

In [ ]:
if condition_cancer == "allBC":

    VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA = pool_dataset_2(datasets,'VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA')
    VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA.group = VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA.metadata
    datasets['VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA'] = VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA
elif condition_cancer == "TN":
    VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA = pool_dataset_2(datasets,'VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA')
    VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA.group = VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA.metadata
    datasets['VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA'] = VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA
        
elif condition_cancer == "all_cancer":
    #BC_pooled
    VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa = pool_dataset_2({'VUMCa': datasets['VUMCa'], 'SRP042620a': datasets['SRP042620a'], 'GSE181466a': datasets['GSE181466a'], 'SRP157974a': datasets['SRP157974a'], 'GSE192341a': datasets['GSE192341a'], 'GSE202203a': datasets['GSE202203a'], 'TCGAa': datasets['TCGAa']},'VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa')
    VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa.group = VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa.metadata
    datasets['VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa'] = VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa
    # delete previous pooled to save memory
    del datasets['VUMCa']
    del datasets['SRP042620a']
    del datasets['GSE181466a']
    del datasets['SRP157974a']
    del datasets['GSE192341a']
    del datasets['GSE202203a']
    del datasets['TCGAa']
    #TN_pooled
    VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA = pool_dataset_2({'VUMC': datasets['VUMC'], 'GSE181466': datasets['GSE181466'], 'SRP157974': datasets['SRP157974'], 'GSE192341': datasets['GSE192341'], 'SRP042620': datasets['SRP042620'], 'GSE202203': datasets['GSE202203'], 'TCGA': datasets['TCGA']},'VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA')
    VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA.group = VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA.metadata
    datasets['VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA'] = VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA
    # delete previous pooled to save memory
    del datasets['VUMC']
    del datasets['GSE181466']
    del datasets['SRP157974']
    del datasets['GSE192341']
    del datasets['SRP042620']
    del datasets['GSE202203']
    del datasets['TCGA']
    
    
elif condition_cancer == "all_cancer_and_BC":
    pass
elif condition_cancer == "allBC+TN":
    #BC_pooled
    VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa = pool_dataset_2({'VUMCa': datasets['VUMCa'], 'SRP042620a': datasets['SRP042620a'], 'GSE181466a': datasets['GSE181466a'], 'SRP157974a': datasets['SRP157974a'], 'GSE192341a': datasets['GSE192341a'], 'GSE202203a': datasets['GSE202203a'], 'TCGAa': datasets['TCGAa']},'VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa')
    VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa.group = VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa.metadata
    datasets['VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa'] = VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa
    #TN_pooled
    VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA = pool_dataset_2({'VUMC': datasets['VUMC'], 'GSE181466': datasets['GSE181466'], 'SRP157974': datasets['SRP157974'], 'GSE192341': datasets['GSE192341'], 'SRP042620': datasets['SRP042620'], 'GSE202203': datasets['GSE202203'], 'TCGA': datasets['TCGA']},'VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA')
    VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA.group = VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA.metadata
    datasets['VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA'] = VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA

else:
    raise ValueError(f"Unknown condition_cancer value: {condition_cancer}")

# 3. Differential expression

In [ ]:
def add_global_batch(dataset_name,dataset):
    """
    Add an adata.obs['batch_global'] column combining:
    - batch_number for TCGA (within-cohort batch)
    - dataset for other cohorts (between-cohort batch)
    """
    
    adata = dataset.anndata
    
    # Safety check: if the column does not exist, set it to 'unknown'.
    if 'batch_number' not in adata.obs.columns:
        adata.obs['batch_number'] = 'unknown'
    
    # Safety check: if COHORT does not exist, use dataset_name.
    if 'COHORT' not in adata.obs.columns:
        adata.obs['COHORT'] = dataset_name
    
    # Create the dataset column if it does not exist.
    if 'dataset' not in adata.obs.columns:
        adata.obs['dataset'] = dataset_name
    
    adata.obs['batch_global'] = np.where(
        adata.obs['COHORT'] == 'TCGA',
        adata.obs['batch_number'],    # batch interne TCGA
        adata.obs['dataset']          # batch = dataset for the other cohorts
    )
    
    return adata

inference = DefaultInference(n_cpus=8)

for dataset_name, dataset in datasets.items():

    print(dataset_name + '*'*50)

    # 1. Add batch_global
    dataset.anndata = add_global_batch(dataset_name,dataset)

    # 2. Choose design formula
    if '_' in dataset_name:  # pooled datasets
        design = ['batch_global', 'condition']
        
    elif dataset_name == 'TCGA':  # TCGA only
        design = ['batch_number', 'condition']

    elif dataset_name=='SRP157974':  # SRP datasets
        design = ['sra.library_selection', 'condition']

    else:  # GEO only, recount only, etc.
        design = ['condition']

    print("→ design_factors =", design)

    # 3. Create the DESeq2 object
    dataset.dds = DeseqDataSet(
        adata=dataset.anndata,
        design_factors=design,
        refit_cooks=True,
        inference=inference,
    )
    
    # 4. Run DESeq2
    # Compute LFCs
    dataset.dds.deseq2()
    # Extract contrast
    dataset.stat_res = DeseqStats(
    dataset.dds,
    contrast=condition_a_tester,
    inference=inference
    )
    # Compute Wald test
    dataset.stat_res.summary()
    # Extract results
    dataset.results_df = dataset.stat_res.results_df
    # Sort by adjusted p-value.
    dataset.results_df = dataset.results_df.sort_values(by='padj')
    dataset.results_df["GeneName"] =  dataset.results_df.index
    #genes up/down
    dataset.genes_up = dataset.results_df[(dataset.results_df['padj'] < 0.05) & (dataset.results_df['log2FoldChange'] > 1)].index
    dataset.genes_down = dataset.results_df[(dataset.results_df['padj'] < 0.05) & (dataset.results_df['log2FoldChange'] < -1)].index


## qq affichages

In [ ]:
# Display dataframes cleanly without print.
from IPython.display import display

for name_dataset, dataset in datasets.items():
    print(name_dataset)
    table = dataset.results_df[dataset.results_df['GeneName'].isin([name_gene,'MTUS1','GSK3B','MYC','HSP90AB1','HSP90AA1','DDX10', 'HYOU1','PSMB5','PSMB6','KAT5','PRMT1','WEE1','BRD4','BAG2', 'FHL2', 'IFI35', 'KIF2A','SDCCAG8', 'SKP1', 'TUBA4A'])].copy()
    # afficher en ordonnant les lignes
    table = table.T[[name_gene,'MTUS1','GSK3B','MYC','HSP90AB1','HSP90AA1','DDX10', 'HYOU1','PSMB5','PSMB6','KAT5','PRMT1','WEE1','BRD4','BAG2', 'FHL2', 'IFI35', 'KIF2A','SDCCAG8', 'SKP1', 'TUBA4A',]]

    display(table.T)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from upsetplot import UpSet

# Example data for eight gene sets.
genes_up = {}
for name_dataset, dataset in datasets.items():
    if name_dataset == 'SUM52':
        name_dataset = 'In vitro data'
    if name_dataset == 'VUMC_GSE192341_GSE123845_GSE202203_TCGA':
        name_dataset = 'All_pulled'
        
    genes_up[name_dataset] = set(dataset.genes_up)



# Create a DataFrame indicating whether each gene is present in each set.
gene_names = set.union(*genes_up.values())
data = {
    gene: {key: (gene in value) for key, value in genes_up.items()}
    for gene in gene_names
}
df = pd.DataFrame.from_dict(data, orient='index')

# Convert to a Series with a MultiIndex for intersections.
# Create a Series whose index is the set MultiIndex and whose values are counts.
df_upset = df.groupby(list(genes_up.keys())).apply(len)


df_upset_series = df_upset.squeeze()  # Convert to Series if needed.

# Configuration et affichage du diagramme UpSet
plt.rcParams.update({'font.size': 14})
upset = UpSet(df_upset_series,  orientation='horizontal', element_size=30, with_lines=True, show_counts='%d') 

upset.plot()

plt.suptitle('Genes upregulated', fontsize=20)
plt.show()




#### see if MYC is up-reglated in different datasets

In [ ]:
for name_dataset, dataset in datasets.items():
    print(name_dataset)
    display(dataset.results_df[dataset.results_df.index == 'MYC'])
    #print('*'*50)

# 4. Enrichment analysis

After performing DEA, we can use the obtained gene level statistics to perform enrichment analysis. Any statistic can be used, but we recommend using the t-values instead of logFCs since t-values incorporate the significance of change in their value. We will transform the obtained t-values stored in stats to a wide matrix so that it can be used by decoupler:

stat = log2FoldChange/lfcSE (standard error value )

 variance (VAR) of each gene by multiplying standard error (SE) by square root of total number of genes (n) and then squaring it :VAR=(SE*sqrt(n))^2

In [ ]:
for dataset in datasets.values():
    dataset.mat = dataset.results_df[['stat']].T.rename(index={'stat': condition_a_tester[1]+'.vs.'+condition_a_tester[2]})

In [ ]:
from gseapy import Msigdb
msig = Msigdb()

# Build a dataframe from a gp.get_library(name='MSigDB_Hallmark_2020') dictionary.
#MSigDB = pd.DataFrame.from_dict(gp.get_library(name='MSigDB_Hallmark_2020'), orient="index") # Previous option; may not be up to date.
#MSigDB = pd.DataFrame.from_dict(gp.get_library(name='WikiPathways_2024_Human'), orient="index") # Alternative option to test.
MSigDB = pd.DataFrame.from_dict(msig.get_gmt(category='h.all', dbver='2024.1.Hs'),orient="index")  # 50 gene sets
# Reshape the table so that gene symbols are in one column and gene-set names in another column.
# List containing all values.
gene_symbols = []
gene_set_names = []
k,l = MSigDB.shape
for i in range(0,k):
    for j in range(0,l):
        gene_symbols.append(MSigDB.iloc[i,j])
        gene_set_names.append(MSigDB.index[i])

# dataframe
MSigDB = pd.DataFrame({'geneset':gene_set_names,'genesymbol':gene_symbols})
# on remove les Noone
MSigDB = MSigDB[MSigDB['genesymbol'].isnull() == False]

In [ ]:

#datasets_5 = {'VUMC':VUMCa, 'GSE192341':GSE192341a,'GSE123845': GSE123845a,  'GSE202203': GSE202203a, 'TCGA': TCGAa,'BC_all_pooled': BC_all_pooled}#,'TNBC_all_pooled': TNBC_all_pooled}

df_pathways = pd.DataFrame()
for dataset_name,dataset in datasets.items():
    top_genes = dataset.results_df[dataset.results_df['padj'].isnull() == False]
    # Restrict to genes in gene_list.
    #top_genes = top_genes[top_genes['GeneName'].isin(list_genes)]

    # Run GSEA
    enr_gsea_pvals = dc.get_gsea_df(
        df=top_genes,
        stat='stat',
        net=MSigDB,
        source='geneset',
        target='genesymbol'
    )
    # Sort by FDR-adjusted p-value.
    enr_gsea_pvals = enr_gsea_pvals.sort_values('FDR p-value')
    dataset.gsea_Hallmark = enr_gsea_pvals
    df_path = dataset.gsea_Hallmark[(dataset.gsea_Hallmark["NOM p-value"] < 0.05) & (dataset.gsea_Hallmark["FDR p-value"] < 0.05)].sort_values(by='NES', ascending=False)
    df_path_2 = df_path[["Term",'NES']]
    #df_path_3 = df_path[["Term",'NES','Leading edge']]
    # Set Term as the index.
    df_path_2.index = df_path_2["Term"]
    # Remove the Term column.
    df_path_2 = df_path_2.drop(columns=["Term"])
    df_path_2 = df_path_2.T
    df_path_2.index = [dataset_name]
    df_pathways = pd.concat([df_pathways,df_path_2],axis=0)

# Draw a heatmap of TF activities.
# Remove columns with more than two NaN values.
df_pathways_2 = df_pathways.dropna(axis=1,thresh=df_pathways.shape[0]-2)
#df_pathways_2 = df_pathways_2[['HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2','HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UV_RESPONSE_DN','HALLMARK_UV_RESPONSE_UP','HALLMARK_UNFOLDED_PROTEIN_RESPONSE','HALLMARK_DNA_REPAIR','HALLMARK_G2M_CHECKPOINT','HALLMARK_ESTROGEN_RESPONSE_EARLY','HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION','HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_XENOBIOTIC_METABOLISM',   'HALLMARK_COAGULATION','HALLMARK_INTERFERON_ALPHA_RESPONSE']]
# Replace NaN values with 0.
df_pathways_2 = df_pathways_2.fillna(0)

plt.figure(figsize=((14,10)))  # Optional: Adjust the size of the heatmap
# Calculate the min and max values for color scaling
vmin = df_pathways_2.min().min()  # Overall minimum value in the DataFrame
vmax = df_pathways_2.max().max()  # Overall maximum value in the DataFrame

ax = sns.heatmap(df_pathways_2.T, annot=True, cmap='coolwarm',  center=0, vmin=vmin, vmax=vmax, fmt='.2f',annot_kws={"fontsize": 15})
plt.xticks(fontsize=14, rotation=90)  # Adjust the fontsize and rotation angle for x-axis
plt.yticks(fontsize=14, rotation=0)   # Adjust the fontsize and rotation angle for y-axis
plt.title('Hallmark pathways, ' + condition_a_tester[1] + ' vs. ' + condition_a_tester[2], fontsize=15)
colorbar = ax.collections[0].colorbar  # Access the color bar object
colorbar.ax.tick_params(labelsize=14)  # Set the fontsize for the color bar labels

plt.show()




In [ ]:
from IPython.display import display
df_pathways = pd.DataFrame()
df_pv = pd.DataFrame()
df_FDR = pd.DataFrame()
for dataset_name,dataset in datasets.items():

    df_path = dataset.gsea_Hallmark.sort_values(by='NES', ascending=False)
    #dataset.gsea_Hallmark[ (dataset.gsea_Hallmark["FDR p-value"] < 0.05)].sort_values(by='NES', ascending=False)
    df_path_2 = df_path[["Term",'NES']]
    df_FDR_2 = df_path[["Term","FDR p-value"]]
    df_pval_2 = df_path[["Term","NOM p-value"]]

    # Set Term as the index.
    df_path_2.index = df_path_2["Term"]
    df_FDR_2.index = df_FDR_2["Term"]
    df_pval_2.index = df_pval_2["Term"]
    # Remove the Term column.
    df_path_2 = df_path_2.drop(columns=["Term"])
    df_path_2 = df_path_2.T
    df_FDR_2 = df_FDR_2.drop(columns=["Term"])
    df_FDR_2 = df_FDR_2.T
    df_pv_2 = df_pval_2.drop(columns=["Term"])
    df_pv_2 = df_pv_2.T

    # Change the index to use dataset names.
    if dataset_name == 'VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA':
        dataset_name = 'TNBC_All_pooled'
    elif dataset_name == 'VUMCa_GSE192341a_GSE123845a_GSE202203a_TCGAa' or dataset_name == 'VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA':
        dataset_name = 'BC_All_pooled'
    if dataset_name == 'SUM-52':
        dataset_name = 'Cell-lines'
    df_path_2.index = [dataset_name]
    df_FDR_2.index = [dataset_name]
    df_pv_2.index = [dataset_name]
    df_pathways = pd.concat([df_pathways,df_path_2],axis=0)
    df_FDR = pd.concat([df_FDR,df_FDR_2],axis=0)
    df_pv = pd.concat([df_pv,df_pv_2],axis=0)

# Create a dataframe with the minimum of df_FDR and df_pv.
df_pval = pd.DataFrame()
for i in range(0,df_FDR.shape[0]):
    for j in range(0,df_FDR.shape[1]):
        if df_FDR.iloc[i,j] > df_pv.iloc[i,j]:
            df_pval.loc[df_FDR.index[i],df_FDR.columns[j]] = df_FDR.iloc[i,j]
        else:
            df_pval.loc[df_FDR.index[i],df_FDR.columns[j]] = df_pv.iloc[i,j]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

col = list(set(df_pathways_2.columns)| set(['HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2', 'HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UV_RESPONSE_DN', 'HALLMARK_DNA_REPAIR','HALLMARK_UV_RESPONSE_UP' ]))
col = ['HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2','HALLMARK_DNA_REPAIR','HALLMARK_E2F_TARGETS', 'HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UNFOLDED_PROTEIN_RESPONSE','HALLMARK_UV_RESPONSE_UP']# 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION','HALLMARK_ESTROGEN_RESPONSE_EARLY','HALLMARK_UV_RESPONSE_DN', 'HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_XENOBIOTIC_METABOLISM','HALLMARK_INTERFERON_GAMMA_RESPONSE','HALLMARK_KRAS_SIGNALING_UP','HALLMARK_INTERFERON_ALPHA_RESPONSE',	'HALLMARK_IL6_JAK_STAT3_SIGNALING' ]

col = ['HALLMARK_MYC_TARGETS_V2','HALLMARK_MYC_TARGETS_V1','HALLMARK_DNA_REPAIR',
       'HALLMARK_ESTROGEN_RESPONSE_EARLY','HALLMARK_UV_RESPONSE_DN','HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION',
       'HALLMARK_XENOBIOTIC_METABOLISM','HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_ANDROGEN_RESPONSE',
       'HALLMARK_E2F_TARGETS', 'HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UNFOLDED_PROTEIN_RESPONSE','HALLMARK_UV_RESPONSE_UP',
       'HALLMARK_BILE_ACID_METABOLISM','HALLMARK_COAGULATION','HALLMARK_MYOGENESIS',
       'HALLMARK_INTERFERON_GAMMA_RESPONSE','HALLMARK_INFLAMMATORY_RESPONSE','HALLMARK_KRAS_SIGNALING_UP','HALLMARK_INTERFERON_ALPHA_RESPONSE', "HALLMARK_ALLOGRAFT_REJECTION",	'HALLMARK_IL6_JAK_STAT3_SIGNALING', "HALLMARK_MITOTIC_SPINDLE"
       ]# 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION','HALLMARK_ESTROGEN_RESPONSE_EARLY','HALLMARK_UV_RESPONSE_DN', 'HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_XENOBIOTIC_METABOLISM','HALLMARK_INTERFERON_GAMMA_RESPONSE','HALLMARK_KRAS_SIGNALING_UP','HALLMARK_INTERFERON_ALPHA_RESPONSE',	'HALLMARK_IL6_JAK_STAT3_SIGNALING' ]


col = ['HALLMARK_MYC_TARGETS_V2','HALLMARK_MYC_TARGETS_V1','HALLMARK_DNA_REPAIR','HALLMARK_E2F_TARGETS','HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UNFOLDED_PROTEIN_RESPONSE', #'HALLMARK_UV_RESPONSE_DN',
       'HALLMARK_UV_RESPONSE_UP']
df_tf_renamed = df_pathways[col]
df_pv_plot = df_pval[col]


# Rename columns by removing "HALLMARK_" from the column names
df_tf_renamed = df_tf_renamed.rename(columns=lambda x: x.replace("HALLMARK_", ""))
#df_tf_renamed = df_tf_renamed.T.rename(columns={'TNBC_All5_pulled':'All_pulled'}).T
df_pv_renamed = df_pv_plot.rename(columns=lambda x: x.replace("HALLMARK_", ""))


# Calculate -log10(p-values)
log_pv = -np.log10(df_pv_renamed.T)
log_pv = log_pv.replace([np.inf, -np.inf], np.nan).fillna(0)

# Define sizes proportional to -log10(p-value)
min_size = 1   # Minimum circle size
max_size = 1000  # Maximum circle size
sizes = log_pv / log_pv.max().max() * (max_size - min_size) + min_size


# Function used to compute circle size from p-values.
def compute_circle_size(p_val):
    if p_val > 0.05:
        return 10  # Very small
    elif p_val > 0.01:
        return 50
    elif p_val > 0.001:
        return 100  # Small
    elif p_val > 0.0001:
        return 200  # Large
    else:
        return 500  # Very large

# Apply the function to each DataFrame element.
sizes =df_pv_renamed.T.applymap(compute_circle_size)

# Create the figure and main axis
fig, ax = plt.subplots(figsize=(8,9))


vmin =  -3 #max(df_tf_renamed.min().min(),-3)  # Overall minimum value in the DataFrame
vmax =  min(df_tf_renamed.max().max(),3)  # Overall maximum value in the DataFrame


# Draw circles.
x_labels =  df_tf_renamed.T.columns
y_labels =  df_tf_renamed.T.index
x_positions = range(len(x_labels))
y_positions = range(len(y_labels))



for y, row in enumerate(y_positions):
    for x, col in enumerate(x_positions):
        value =  df_tf_renamed.T.iloc[y, x]
        size = sizes.iloc[y, x]
        #print(f"value: {value}, vmin: {vmin}, vmax: {vmax}")
        #color = plt.cm.Reds((value - vmin) / (vmax - vmin))
        color = plt.cm.coolwarm((value - vmin) / (vmax - vmin)) 
        ax.scatter(x, -y, s=size, c=[color], edgecolor='k', alpha=0.8)

# Adjust axes and labels
ax.set_xticks(x_positions)
ax.set_xticklabels(x_labels, rotation=90)
ax.set_yticks([-i for i in y_positions])
ax.set_yticklabels(y_labels)
ax.invert_yaxis()
ax.set_aspect('equal')
ax.set_xlim(-0.5, len(x_labels) - 0.5)
ax.set_ylim(-len(y_labels) + 0.5, 0.5)

legend_sizes = [10, 50, 100, 200,500]  # Example sizes.
legend_labels = ["p > 0.05", "0.01 < p ≤ 0.05", "0.001 < p ≤ 0.01", "0.0001 < p ≤ 0.001", "p ≤ 0.0001"]
legend_circles = [plt.scatter([], [], s=size, color='gray', edgecolor='k', alpha=0.8) for size in legend_sizes]

# Create an axis for the legend.
#ax_legend1  = fig.add_axes([0.87, 0.2, 0.05, 0.2])  # Position (gauche, bas, largeur, hauteur)
#ax_legend1.axis("off")  # Hide unused axes.
#ax_legend1.legend(legend_circles, legend_labels, title="Adjusted p_value", loc="center", frameon=True)


# Add colorbar for NES below the plot
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical', pad=0.05, fraction=0.02, aspect=15)
cbar.set_label("NES", fontsize=15)


# Add the global title at the top.
#fig.suptitle("Pathways NES Scores, summary of all datasets", fontsize=16, fontweight='bold')

# Refine layout to avoid overlaps and properly align legends
plt.subplots_adjust(bottom=.65, top=0.9)
plt.show()



In [ ]:
df_tf_renamed

In [ ]:
df_pv_renamed 